# Enterprise Project 2 - Real-Time IT Operations A2A Copilot

This notebook builds a real-time IT operations Copilot that uses Azure AI Foundry Agents, OpenAPI API tools, and Agent-to-Agent (A2A) integration. It exposes a Foundry-backed SRE agent as an A2A service so another agent, workflow, or Copilot client can call it.

## Business scenario

An enterprise platform team wants an SRE Copilot that can inspect live service-health signals, combine them with CMDB or change-management APIs, and publish the capability to other agents using A2A.

## Architecture

1. Foundry SRE agent: understands incidents, SLO impact, and mitigation runbooks.
2. API tools: calls public GitHub status and optional enterprise ITSM/CMDB endpoints.
3. A2A server: exposes the SRE agent through the A2A protocol.
4. A2A client: simulates another Copilot or workflow invoking the SRE agent.
5. Real-time stream: sends live incident prompts and streams response deltas.

> Production note: if you expose this beyond localhost, add authentication, authorization, request limits, trace logging, and network controls.


## 1. Install Required Packages

The notebook uses the same A2A SDK pattern as the repo examples, plus Foundry Agent SDK packages.


In [ ]:
%pip install a2a-sdk==0.3.8 azure-ai-projects==2.0.0b2 openai==1.109.1 python-dotenv azure-identity jsonref httpx uvicorn rich


## 2. Load Configuration

Required `.env` values:

```text
FOUNDRY_PROJECT_ENDPOINT=https://<your-project>.services.ai.azure.com/api/projects/<project-name>
MODEL_DEPLOYMENT_NAME=<your-model-deployment>
```

Optional enterprise API value:

```text
ENTERPRISE_IT_API_BASE_URL=https://<your-it-api-host>
```


In [ ]:
import os
import json
import asyncio
import random
import uuid
import threading
from datetime import datetime, timezone
from typing import Any, AsyncGenerator

import httpx
import jsonref
import uvicorn
from dotenv import load_dotenv
from rich.console import Console
from rich.panel import Panel
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition

load_dotenv()
console = Console()

foundry_project_endpoint = os.getenv("FOUNDRY_PROJECT_ENDPOINT")
model_deployment_name = os.getenv("MODEL_DEPLOYMENT_NAME")
enterprise_it_api_base_url = os.getenv("ENTERPRISE_IT_API_BASE_URL", "").rstrip("/")
sre_copilot_max_retries = int(os.getenv("SRE_COPILOT_MAX_RETRIES", "3"))
sre_copilot_max_output_tokens = int(os.getenv("SRE_COPILOT_MAX_OUTPUT_TOKENS", "800"))

def require_env(name: str, value: str | None) -> str:
    if not value:
        raise ValueError(f"Missing required environment variable: {name}")
    return value

foundry_project_endpoint = require_env("FOUNDRY_PROJECT_ENDPOINT", foundry_project_endpoint)
model_deployment_name = require_env("MODEL_DEPLOYMENT_NAME", model_deployment_name)

print(f"Project endpoint: {foundry_project_endpoint}")
print(f"Model deployment: {model_deployment_name}")
print(f"Enterprise IT API configured: {bool(enterprise_it_api_base_url)}")
print(f"SRE Copilot max retries: {sre_copilot_max_retries}")
print(f"SRE Copilot max output tokens: {sre_copilot_max_output_tokens}")


## 3. Create Foundry Clients

The A2A executor will use the Foundry agent through the OpenAI-compatible Responses API.


In [ ]:
project_client = AIProjectClient(endpoint=foundry_project_endpoint, credential=DefaultAzureCredential())
openai_client = project_client.get_openai_client()
print("Foundry clients are ready.")


## 4. Define API Tool Specifications

The GitHub status API is public and useful for a live external-health example. The optional enterprise IT API shows where to plug in CMDB, incident, change, or service dependency data.


In [ ]:
github_status_spec: dict[str, Any] = {
    "openapi": "3.1.0",
    "info": {"title": "GitHub Status API", "description": "Public GitHub service status API for live operational health checks.", "version": "1.0.0"},
    "servers": [{"url": "https://www.githubstatus.com"}],
    "paths": {
        "/api/v2/status.json": {"get": {"operationId": "GetGitHubComponentStatus", "description": "Get the current high-level GitHub service status.", "responses": {"200": {"description": "Current status", "content": {"application/json": {"schema": {"type": "object"}}}}}}},
        "/api/v2/incidents.json": {"get": {"operationId": "GetGitHubIncidents", "description": "Get recent GitHub incidents.", "responses": {"200": {"description": "Recent incidents", "content": {"application/json": {"schema": {"type": "object"}}}}}}},
    },
}

enterprise_it_spec: dict[str, Any] | None = None
if enterprise_it_api_base_url:
    enterprise_it_spec = {
        "openapi": "3.1.0",
        "info": {"title": "Enterprise IT Operations API", "description": "CMDB, incident, service dependency, and change window lookup API.", "version": "1.0.0"},
        "servers": [{"url": enterprise_it_api_base_url}],
        "paths": {
            "/services/{service_id}/dependencies": {"get": {"operationId": "GetServiceDependencies", "description": "Get upstream and downstream dependencies for a service.", "parameters": [{"name": "service_id", "in": "path", "required": True, "schema": {"type": "string"}}], "responses": {"200": {"description": "Dependencies", "content": {"application/json": {"schema": {"type": "object"}}}}}}},
            "/changes/active": {"get": {"operationId": "GetActiveChangeWindows", "description": "Get active or recently completed change windows.", "responses": {"200": {"description": "Active changes", "content": {"application/json": {"schema": {"type": "object"}}}}}}},
            "/incidents/{incident_id}": {"get": {"operationId": "GetIncidentRecord", "description": "Get an incident record from the enterprise ITSM system.", "parameters": [{"name": "incident_id", "in": "path", "required": True, "schema": {"type": "string"}}], "responses": {"200": {"description": "Incident record", "content": {"application/json": {"schema": {"type": "object"}}}}}}},
        },
    }

api_tools = [{"type": "openapi", "openapi": {"name": "github_status", "spec": jsonref.loads(json.dumps(github_status_spec)), "auth": {"type": "anonymous"}}}]
if enterprise_it_spec:
    api_tools.append({"type": "openapi", "openapi": {"name": "enterprise_it_ops", "spec": jsonref.loads(json.dumps(enterprise_it_spec)), "auth": {"type": "anonymous"}}})

print(f"Prepared {len(api_tools)} API tool definition(s).")


## 5. Create the SRE Foundry Agent

The agent is optimized for incident diagnosis, blast-radius analysis, and executive-safe updates.


In [ ]:
sre_agent_name = "ent-itops-a2a-sre-agent"

sre_instructions = """
You are an enterprise SRE Copilot exposed through Agent-to-Agent integration.

Your job:
- Analyze service incidents, dependency risk, change collisions, and customer impact.
- Use API tools for live service status or enterprise IT context when relevant.
- Return concise, operationally useful guidance.
- Be explicit about confidence and missing evidence.
- Recommend mitigation steps that can be safely handed to an on-call engineer.

Response format:
1. Situation summary
2. Current health and evidence
3. Likely blast radius
4. Mitigation plan
5. Escalation or approval needed
6. Message suitable for a stakeholder update
""".strip()

sre_agent = project_client.agents.create_version(
    agent_name=sre_agent_name,
    definition=PromptAgentDefinition(model=model_deployment_name, instructions=sre_instructions, tools=api_tools),
)

print(f"SRE agent created (id: {sre_agent.id}, name: {sre_agent.name}, version: {sre_agent.version})")


## 6. Wrap the Foundry Agent for A2A

This class streams tokens from the Foundry Responses API. The A2A server will call this wrapper whenever another agent sends a task.


In [ ]:
def is_transient_capacity_error(exc: Exception) -> bool:
    message = str(exc).lower()
    transient_markers = [
        "high demand",
        "peak load",
        "rate limit",
        "too many requests",
        "temporarily unavailable",
        "server overloaded",
        "429",
        "503",
    ]
    return any(marker in message for marker in transient_markers)


class FoundrySreAgent:
    """Adapter that invokes the Foundry SRE agent and streams text deltas."""

    async def invoke_stream(self, user_query: str) -> AsyncGenerator[dict[str, Any], None]:
        last_error: Exception | None = None
        for attempt in range(1, sre_copilot_max_retries + 2):
            try:
                conversation = openai_client.conversations.create()
                response_stream = openai_client.responses.create(
                    conversation=conversation.id,
                    extra_body={"agent": {"name": sre_agent_name, "type": "agent_reference"}},
                    input=user_query,
                    max_output_tokens=sre_copilot_max_output_tokens,
                    stream=True,
                )
                for event in response_stream:
                    if event.type == "response.output_text.delta":
                        yield {"content": event.delta, "done": False}
                yield {"content": "", "done": True}
                return
            except Exception as exc:
                last_error = exc
                if attempt > sre_copilot_max_retries or not is_transient_capacity_error(exc):
                    break

                delay_seconds = min(2 ** attempt + random.random(), 12)
                yield {"content": f"SRE Copilot hit temporary model capacity limits. Retrying in {delay_seconds:.1f}s...\n", "done": False}
                await asyncio.sleep(delay_seconds)

        guidance = "Try again shortly, reduce SRE_COPILOT_MAX_OUTPUT_TOKENS, or switch MODEL_DEPLOYMENT_NAME to a deployment with more reliable capacity. For production demos, use Provisioned Throughput."
        yield {"content": f"SRE Copilot failed to process request after retries: {last_error}\n\n{guidance}", "done": True}


## 7. Build the A2A Executor, Card, and Server

The Agent Card describes the capability so another agent or Copilot can discover and call it.


In [ ]:
from a2a.server.agent_execution import AgentExecutor, RequestContext
from a2a.server.events import EventQueue
from a2a.server.request_handlers import DefaultRequestHandler
from a2a.server.tasks import InMemoryTaskStore
from a2a.server.apps import A2AStarletteApplication
from a2a.types import AgentCapabilities, AgentCard, AgentSkill, TaskArtifactUpdateEvent, TaskState, TaskStatus, TaskStatusUpdateEvent
from a2a.utils import new_text_artifact

class SreAgentExecutor(AgentExecutor):
    """A2A executor that delegates work to the Foundry SRE agent."""

    def __init__(self) -> None:
        self.agent = FoundrySreAgent()

    async def execute(self, context: RequestContext, event_queue: EventQueue) -> None:
        query = context.get_user_input()
        if not context.message:
            raise ValueError("No A2A message provided")

        async for event in self.agent.invoke_stream(query):
            await event_queue.enqueue_event(TaskArtifactUpdateEvent(context_id=context.context_id, task_id=context.task_id, artifact=new_text_artifact(name="sre_copilot_response", text=event["content"])))
            if event["done"]:
                break

        await event_queue.enqueue_event(TaskStatusUpdateEvent(context_id=context.context_id, task_id=context.task_id, status=TaskStatus(state=TaskState.completed), final=True))

    async def cancel(self, context: RequestContext, event_queue: EventQueue) -> None:
        raise ValueError("Cancel is not supported in this demo")

skill = AgentSkill(
    id="sre_incident_triage",
    name="SRE Incident Triage",
    description="Analyze incidents, service health, dependency risk, and mitigation options.",
    tags=["sre", "itops", "incident", "foundry", "api-tools"],
    examples=["Triage incident INC-1001 for checkout latency.", "Check live external status and draft an executive update."],
)

public_agent_card = AgentCard(
    name="Enterprise IT Operations SRE Copilot",
    description="A2A endpoint backed by an Azure AI Foundry Agent with API tool grounding.",
    url="http://localhost:8088",
    version="1.0.0",
    default_input_modes=["text"],
    default_output_modes=["text"],
    capabilities=AgentCapabilities(streaming=True),
    skills=[skill],
)

request_handler = DefaultRequestHandler(agent_executor=SreAgentExecutor(), task_store=InMemoryTaskStore())
a2a_app = A2AStarletteApplication(agent_card=public_agent_card, http_handler=request_handler)
print("A2A app created. Agent card will be available at http://localhost:8088/.well-known/agent.json")


## 8. Start the Local A2A Server

This starts the server in a background thread so the next notebook cells can act as an A2A client. Stop the kernel to stop the server.


In [ ]:
def run_a2a_server() -> None:
    config = uvicorn.Config(a2a_app.build(), host="0.0.0.0", port=8088, loop="asyncio", log_level="warning")
    server = uvicorn.Server(config)
    server.run()

server_thread = threading.Thread(target=run_a2a_server, daemon=True)
server_thread.start()
print("A2A server starting on http://localhost:8088")


## 9. Create an A2A Client

This client simulates another Copilot or workflow calling the SRE agent. It resolves the agent card, creates a JSON-RPC transport client, and streams the returned artifacts.


In [ ]:
from a2a.client import ClientFactory, create_text_message_object
from a2a.client.card_resolver import A2ACardResolver
from a2a.client.client import ClientConfig
from a2a.types import AgentCard, TransportProtocol
from a2a.utils.message import get_message_text

async def call_sre_a2a_agent(prompt: str, base_url: str = "http://localhost:8088") -> str:
    chunks: list[str] = []
    last_artifact_text: str | None = None
    async with httpx.AsyncClient(timeout=httpx.Timeout(connect=10.0, read=90.0, write=10.0, pool=10.0)) as httpx_client:
        resolver = A2ACardResolver(httpx_client=httpx_client, base_url=base_url)
        agent_card: AgentCard = await resolver.get_agent_card()
        config = ClientConfig(httpx_client=httpx_client, supported_transports=[TransportProtocol.jsonrpc], streaming=True)
        client = ClientFactory(config).create(agent_card)
        request = create_text_message_object(content=prompt)

        async for response in client.send_message(request):
            task, _ = response
            if task.artifacts:
                text = get_message_text(task.artifacts[-1])
                if text != last_artifact_text:
                    chunks.append(text)
                    print(text, end="", flush=True)
                    last_artifact_text = text
    return "".join(chunks)


## 10. Send a Real-Time Incident Prompt Over A2A

This event looks like something another monitoring agent could emit after detecting elevated latency.


In [ ]:
incident_event = {
    "event_id": str(uuid.uuid4()),
    "event_type": "slo_breach_detected",
    "occurred_at": datetime.now(timezone.utc).isoformat(),
    "service_id": "checkout-api",
    "severity": "high",
    "symptoms": ["p95 latency above 2200 ms for 12 minutes", "payment authorization retries increased by 18 percent", "customer checkout completion rate down 6 percent"],
    "suspected_dependencies": ["payment-gateway", "feature-flag-service", "github-actions-deployment"],
}

prompt = f"""
You are receiving a real-time incident from an upstream monitoring Copilot.
Analyze it, use API tools if useful, and return mitigation guidance.

Incident event:
{json.dumps(incident_event, indent=2)}
""".strip()

console.print(Panel(prompt, title="A2A Request"))
response_text = await call_sre_a2a_agent(prompt)


## 11. Operationalize This Pattern

Recommended enterprise upgrades:

- Put the A2A service behind Azure Container Apps, AKS, or App Service.
- Add Entra ID or signed service-to-service tokens before exposing the endpoint.
- Store every A2A request and response with `event_id`, `task_id`, and `context_id`.
- Add tool-level allowlists for high-risk actions.
- Wire stakeholder updates to Teams, ServiceNow, PagerDuty, or Azure Monitor action groups.
- Add regression evaluations for outage handling, escalation correctness, and unsafe-action prevention.
